# Install

In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', '-q'], returncode=0)

# Khởi tạo SparkSession

In [2]:
import os
import io
import sys
import pandas as pd
from pyspark.sql import SparkSession
from azure.storage.blob import BlobServiceClient
 
ACCOUNT_NAME = os.environ["AZURE_STORAGE_ACCOUNT_NAME"]
ACCOUNT_KEY  = os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
CONTAINER    = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")
CONN_STR     = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
 
spark = (SparkSession.builder
    .appName("Fix_Artifacts")
    .master("spark://spark-master:7077")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.3.4")
    .config(f"spark.hadoop.fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net", ACCOUNT_KEY)
    .getOrCreate())
 
BASE_PATH = f"wasbs://{CONTAINER}@{ACCOUNT_NAME}.blob.core.windows.net"
blob_service = BlobServiceClient.from_connection_string(CONN_STR)
container_client = blob_service.get_container_client(CONTAINER)
 
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"BASE_PATH: {BASE_PATH}")

Spark version: 4.0.0
Master: spark://spark-master:7077
BASE_PATH: wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net


# Hàm fix 2 file free_flow_speed_lookup.parquet + link_coordinates.parquet

In [3]:
def fix_artifact(artifact_name: str):
    # artifact_name: ví dụ 'artifacts/free_flow_speed_lookup.parquet'
    # Đọc lại data từ folder Spark đã ghi sai (chứa _SUCCESS + part-00000...),
    # convert sang pandas, xóa blob/folder rác cũ, ghi lại thành 1 file parquet đúng.
    print(f"\nFixing {artifact_name}")
 
    folder_path = f"{BASE_PATH}/{artifact_name}"
 
    # 1. Đọc data hiện có từ folder Spark
    df_spark = spark.read.parquet(folder_path)
    pdf = df_spark.toPandas()
    print(f"Đọc được {len(pdf)} rows, columns: {list(pdf.columns)}")
 
    if len(pdf) == 0:
        print(f"CẢNH BÁO: {artifact_name} đọc ra 0 rows. KHÔNG xóa gì, dừng lại kiểm tra thủ công.")
        return
 
    # 2. Convert sang 1 file parquet đơn (bytes)
    buf = io.BytesIO()
    pdf.to_parquet(buf, index=False, engine="pyarrow")
    buf.seek(0)
    new_file_bytes = buf.getvalue()
    print(f"File parquet mới: {len(new_file_bytes)} bytes")
 
    # 3. Xóa blob 0-byte cùng tên + mọi blob trong folder rác (part-files, _SUCCESS)
    deleted = []
    try:
        container_client.delete_blob(artifact_name)
        deleted.append(artifact_name)
    except Exception as e:
        print(f"  (bỏ qua xóa {artifact_name}: {e})")
 
    prefix = artifact_name + "/"
    for blob in container_client.list_blobs(name_starts_with=prefix):
        container_client.delete_blob(blob.name)
        deleted.append(blob.name)
 
    print(f"Đã xóa {len(deleted)} blob rác: {deleted}")
 
    # 4. Upload file parquet đúng lên đúng path
    blob_client = blob_service.get_blob_client(container=CONTAINER, blob=artifact_name)
    blob_client.upload_blob(new_file_bytes, overwrite=True)
    print(f"Đã ghi lại {artifact_name} ({len(new_file_bytes)} bytes) - FIXED")

# Chạy hàm fix

In [4]:
fix_artifact("artifacts/free_flow_speed_lookup.parquet")
fix_artifact("artifacts/link_coordinates.parquet")
 
print("\nXONG. Kiểm tra lại trên Azure Storage Explorer:")
print("artifacts/free_flow_speed_lookup.parquet  -> phải là 1 file, size > 0")
print("artifacts/link_coordinates.parquet -> phải là 1 file, size > 0")


Fixing artifacts/free_flow_speed_lookup.parquet
Đọc được 116 rows, columns: ['link_id', 'free_flow_speed', 'borough', 'link_name']
File parquet mới: 6875 bytes
Đã xóa 3 blob rác: ['artifacts/free_flow_speed_lookup.parquet', 'artifacts/free_flow_speed_lookup.parquet/_SUCCESS', 'artifacts/free_flow_speed_lookup.parquet/part-00000-84ed8fc3-bfc3-4b70-b19e-969476eeafe4-c000.snappy.parquet']
Đã ghi lại artifacts/free_flow_speed_lookup.parquet (6875 bytes) - FIXED

Fixing artifacts/link_coordinates.parquet
Đọc được 125 rows, columns: ['link_id', 'lat', 'lon']
File parquet mới: 5022 bytes
Đã xóa 95 blob rác: ['artifacts/link_coordinates.parquet', 'artifacts/link_coordinates.parquet/_SUCCESS', 'artifacts/link_coordinates.parquet/part-00000-ec51fa16-6f9c-474a-8df1-87e1bb5967de-c000.snappy.parquet', 'artifacts/link_coordinates.parquet/part-00001-ec51fa16-6f9c-474a-8df1-87e1bb5967de-c000.snappy.parquet', 'artifacts/link_coordinates.parquet/part-00002-ec51fa16-6f9c-474a-8df1-87e1bb5967de-c000.snap

# Verify lại 2 file trên

In [7]:
import io
import pandas as pd
from azure.storage.blob import BlobServiceClient

# Dùng lại CONN_STR, CONTAINER đã có từ cell trước
blob_service_check = BlobServiceClient.from_connection_string(CONN_STR)
container_client = blob_service_check.get_container_client(CONTAINER)

def verify_single_blob_parquet(blob_path: str, expected_rows: int = None, expected_cols=None):
    print(f"VERIFY: {blob_path}")

    blob_client = container_client.get_blob_client(blob_path)

    # 1. Check blob tồn tại + lấy properties (size, metadata)
    if not blob_client.exists():
        print(f"FAIL — blob không tồn tại: {blob_path}")
        return False, None

    props = blob_client.get_blob_properties()
    size_bytes = props.size
    is_folder_marker = props.metadata.get("hdi_isfolder") if props.metadata else None

    print(f"Size: {size_bytes} bytes")
    print(f"Metadata hdi_isfolder: {is_folder_marker}")

    if size_bytes == 0:
        print(f"FAIL — file 0 byte (vẫn là placeholder/rác)")
        return False, None

    if is_folder_marker == "true":
        print(f"FAIL — đây là folder marker, không phải file dữ liệu thật")
        return False, None

    # 2. Check không còn folder rác cùng tên (path/part-00000...)
    folder_prefix = blob_path + "/"
    leftover = list(container_client.list_blobs(name_starts_with=folder_prefix))
    if leftover:
        print(f"WARNING — vẫn còn {len(leftover)} blob rác trong folder cùng tên:")
        for b in leftover:
            print(f"    - {b.name} ({b.size} bytes)")
    else:
        print(f"OK — không còn folder rác cùng tên")

    # 3. Đọc thử bằng pandas/pyarrow để chắc chắn parse được
    try:
        raw = blob_client.download_blob().readall()
        df = pd.read_parquet(io.BytesIO(raw))
    except Exception as e:
        print(f"FAIL — không đọc được bằng pandas/pyarrow: {e}")
        return False, None

    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(5))

    ok = True
    if expected_rows is not None and len(df) != expected_rows:
        print(f"WARNING — kỳ vọng {expected_rows} rows nhưng đọc được {len(df)} rows")
        ok = False
    if expected_cols is not None:
        missing = set(expected_cols) - set(df.columns)
        if missing:
            print(f"WARNING — thiếu cột: {missing}")
            ok = False
    if ok:
        print(f"PASS — {blob_path} hợp lệ")
    return ok, df

# Chạy verify cho cả 2 file
ok1, df_lookup = verify_single_blob_parquet(
    "artifacts/free_flow_speed_lookup.parquet",
    expected_rows=116,
    expected_cols=["link_id", "free_flow_speed", "borough", "link_name"],
)

ok2, df_coords = verify_single_blob_parquet(
    "artifacts/link_coordinates.parquet",
    expected_rows=125,
    expected_cols=["link_id", "lat", "lon"],
)

print(f"free_flow_speed_lookup.parquet : {'PASS' if ok1 else 'FAIL/WARNING'}")
print(f"link_coordinates.parquet       : {'PASS' if ok2 else 'FAIL/WARNING'}")

if ok1 and ok2:
    print("\nCả 2 file artifact đã đúng định dạng — có thể chạy lại Job A/Job B.")
else:
    print("\nCần kiểm tra lại — xem chi tiết log phía trên (có thể vẫn còn folder rác hoặc sai schema).")

VERIFY: artifacts/free_flow_speed_lookup.parquet
Size: 6875 bytes
Metadata hdi_isfolder: None
OK — không còn folder rác cùng tên
Rows: 116
Columns: ['link_id', 'free_flow_speed', 'borough', 'link_name']
   link_id  free_flow_speed        borough  \
0  4362251            53.43         Queens   
1  4616192            58.40  Staten Island   
2  4616310            48.46          Bronx   
3  4616246            59.03          Bronx   
4  4616241            55.30         Queens   

                                           link_name  
0                   LIE WB LITTLE NECK PKWY - NB CVE  
1                SIE E CLOVE ROAD - FINGERBOARD ROAD  
2             CBE AMSTERDAM AVE (L/LVL) - MORRIS AVE  
3            BE N STRATFORD AVENUE - CASTLE HILL AVE  
4  VWE S MP8.65 (Exit 13 Northern Blvd) - MP6.39 ...  
PASS — artifacts/free_flow_speed_lookup.parquet hợp lệ
VERIFY: artifacts/link_coordinates.parquet
Size: 5022 bytes
Metadata hdi_isfolder: None
OK — không còn folder rác cùng tên
Rows: 125
Co

# Kiểm tra các cột

In [8]:
print(df_lookup.dtypes)
print(df_coords.dtypes)

link_id             object
free_flow_speed    float64
borough             object
link_name           object
dtype: object
link_id     object
lat        float64
lon        float64
dtype: object
